## Uses dataframe and verb/word locations to add word spans for label studio

Uses koondkorpus_examples.csv data file.

In [1]:
import pandas as pd
from estnltk import Text
from estnltk.taggers.system.rule_taggers.extraction_rules.ruleset import Ruleset
from estnltk.taggers.system.rule_taggers.extraction_rules.static_extraction_rule import StaticExtractionRule
from estnltk.taggers.system.rule_taggers.taggers.substring_tagger import SubstringTagger
from estnltk.converters.label_studio.labelling_configurations import PhraseTaggingConfiguration
from estnltk.converters.label_studio.labelling_tasks import PhraseTaggingTask

## Configuration


In [2]:
SOURCE_DIR = "../source_data"
DATA_FILE = f"{SOURCE_DIR}/koondkorpus_examples.csv"

## Read in data

In [3]:
data = pd.read_csv(DATA_FILE, sep=";", encoding="utf-8")

In [4]:
data

,head_id,sentence_id,verb,verb_compound,verb_loc,verb_form,root_deprel,kaane,root_lemma,root_form,root_loc,root_loc_rel,root_parent_loc,koht,elus,sentence
0,875,485,tulema,NaN,6,tulen,obl,adit,kodu,koju,5,-1,NaN,YES,NaN,"Niipea , kui ma koju tulen , muutub ta tujukaks ja võimukaks : “ Tee seda !"
1,731,396,tulema,NaN,12,tuleb,obl,adit,kodu,koju,13,1,NaN,YES,NaN,Milline seksuaalfantaasia tundub kõige apetiitsem : seks tundmatuga liftis ; torulukksepp tuleb koju ; jõuluvana ja snegurotška või doktor ja patsient ?
2,53,39,tulema,NaN,5,tuled,obl,adit,toim,toime,4,-1,NaN,NaN,NaN,Kuidas sa rahaliselt toime tuled ?
3,497,264,tulema,NaN,12,tulnud,obl,adit,kontserdimaja,kontserdimajja,14,1,NaN,NaN,NaN,"Mäe on kohtunud piletikassa juures pärnakatega , kes väidavad end olevat tulnud uude kontserdimajja juba üheksandat-kümnendat korda ."


## Workflow

### EstNLTK task

In [19]:
sentences = []
for i in range(len(data)):
    sentence = Text(data.iloc[i]["sentence"])
    rules = Ruleset([
        StaticExtractionRule(data.iloc[i]["verb_form"], {'label': 'verb'}),
        StaticExtractionRule(data.iloc[i]["root_form"], {'label': 'nimisõna'}),
    ])
    tagger = SubstringTagger(rules, output_attributes=['label'], ignore_case=True)
    tagger.tag(sentence)
    sentences.append(sentence)
    
    #sentence.terms.display()

Niipea , kui ma koju tulen , muutub ta tujukaks ja võimukaks : “ Tee seda !

Milline seksuaalfantaasia tundub kõige apetiitsem : seks tundmatuga liftis ; torulukksepp tuleb koju ; jõuluvana ja snegurotška või doktor ja patsient ?

Kuidas sa rahaliselt toime tuled ?

Mäe on kohtunud piletikassa juures pärnakatega , kes väidavad end olevat tulnud uude kontserdimajja juba üheksandat-kümnendat korda .

In [26]:
conf = PhraseTaggingConfiguration(['verb', 'nimisõna'], header="Verbobl ja nimisõnafraas")

In [27]:
task = PhraseTaggingTask(conf, input_layer=tagger.output_layer, 
                         output_layer='annotated_partofspeech', 
                         label_attribute='label')

In [28]:
print(task.interface_file)

<View>
  <Header value="Verbobl ja nimisõnafraas" />
  <Labels name="annotated_partofspeech" toName="text" >
    <Label value="verb" background="green" />
    <Label value="nimisõna" background="blue" />
  </Labels>
  <Text name="text" value="$text" granularity="word" />
</View>


In [29]:
print(task.export_data(sentences, indent=2))

[
  {
    "data": {
      "text": "Niipea , kui ma koju tulen , muutub ta tujukaks ja v\u00f5imukaks : \u201c Tee seda !"
    },
    "annotations": [
      {
        "result": [
          {
            "value": {
              "start": 16,
              "end": 20,
              "labels": [
                "nimis\u00f5na"
              ]
            },
            "from_name": "annotated_partofspeech",
            "to_name": "text",
            "type": "labels"
          },
          {
            "value": {
              "start": 21,
              "end": 26,
              "labels": [
                "verb"
              ]
            },
            "from_name": "annotated_partofspeech",
            "to_name": "text",
            "type": "labels"
          }
        ]
      }
    ]
  },
  {
    "data": {
      "text": "Milline seksuaalfantaasia tundub k\u00f5ige apetiitsem : seks tundmatuga liftis ; torulukksepp tuleb koju ; j\u00f5uluvana ja snegurot\u0161ka v\u00f5i doktor ja patsient

Save label studio data file

In [30]:
with open('labelstudio/koondkorpus_examples_v1.json', 'w') as f:
    f.write( task.export_data(sentences, indent=2) )